# Member B: Decision Tree + Random Forest + eli5/LIME Interpretability
## CS667 Project 4: ML Interpretability

**Assigned Tasks:**
- EDA Focus: Feature distributions
- Models: Decision Tree + Random Forest with GridSearchCV
- Interpretability: eli5 for DT (Task 3_B) + LIME for RF (Task 3_C partial)

**Prerequisites:** Run `00_data_preparation.ipynb` first!

---
# Part 1: Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
import joblib

import eli5
from eli5.sklearn import PermutationImportance

import lime
import lime.lime_tabular

# Shared constants
RANDOM_STATE = 42
TARGET_COLUMN = 'DEATH_EVENT'

In [ ]:
# Load the pre-split data
train_data = pd.read_csv('../data/train.csv')
test_data = pd.read_csv('../data/test.csv')

X_train = train_data.drop(columns=[TARGET_COLUMN])
y_train = train_data[TARGET_COLUMN]
X_test = test_data.drop(columns=[TARGET_COLUMN])
y_test = test_data[TARGET_COLUMN]

feature_names = X_train.columns.tolist()

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Features: {feature_names}")

---
# Part 2: Exploratory Data Analysis

**Your EDA Focus**: Feature distributions (univariate analysis).

### Investigation Prompts:
> - How are continuous features distributed? Are there outliers?
> - What is the distribution of binary features across the two classes?
> - Do any features show clear separation between death/survival groups?

In [ ]:
# Define feature types
BINARY_FEATURES = ['anaemia', 'diabetes', 'high_blood_pressure', 'sex', 'smoking']
CONTINUOUS_FEATURES = ['age', 'creatinine_phosphokinase', 'ejection_fraction', 
                       'platelets', 'serum_creatinine', 'serum_sodium', 'time']

In [ ]:
# TODO: Create histograms for continuous features, colored by target
# Hint: Use train_data for EDA, with hue=TARGET_COLUMN

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, feature in enumerate(CONTINUOUS_FEATURES):
    # YOUR CODE HERE
    pass

plt.tight_layout()
plt.savefig('../visualizations/continuous_distributions.png', dpi=150)
plt.show()

In [ ]:
# TODO: Create bar plots for binary features, showing proportions by target
# Hint: Crosstab or groupby can help calculate proportions

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, feature in enumerate(BINARY_FEATURES):
    # YOUR CODE HERE
    pass

plt.tight_layout()
plt.savefig('../visualizations/binary_distributions.png', dpi=150)
plt.show()

### EDA Findings:

*Document your key observations:*

1. Features showing clear separation between classes: ...
2. Outliers detected in: ...
3. Skewed distributions: ...
4. Implications for tree-based models: ...

---
# Part 3: Decision Tree Modeling

### Investigation Prompts:
> - What hyperparameters control Decision Tree complexity?
> - What is the risk of overfitting with deep trees?
> - Why don't trees require feature scaling?

In [ ]:
# TODO: Define the hyperparameter grid for Decision Tree
# Consider: max_depth, min_samples_split, min_samples_leaf, criterion

dt_param_grid = {
    # YOUR CODE HERE
    # Hint: 'max_depth': [3, 5, 7, 10, None]
    # Hint: 'min_samples_split': [2, 5, 10]
    # Hint: 'min_samples_leaf': [1, 2, 4]
}

print(f"DT Parameter grid: {dt_param_grid}")

In [ ]:
# TODO: Perform GridSearchCV for Decision Tree

dt = DecisionTreeClassifier(random_state=RANDOM_STATE)

dt_grid_search = GridSearchCV(
    # YOUR CODE HERE
)

# Fit
# YOUR CODE HERE

print(f"Best DT parameters: {dt_grid_search.best_params_}")
print(f"Best DT CV AUC-ROC: {dt_grid_search.best_score_:.4f}")

In [ ]:
# Evaluate Decision Tree on test set
best_dt = dt_grid_search.best_estimator_

y_pred_dt = best_dt.predict(X_test)
y_proba_dt = best_dt.predict_proba(X_test)[:, 1]

dt_accuracy = accuracy_score(y_test, y_pred_dt)
dt_auc_roc = roc_auc_score(y_test, y_proba_dt)

print(f"\n=== Decision Tree Results ===")
print(f"Test Accuracy: {dt_accuracy:.4f}")
print(f"Test AUC-ROC: {dt_auc_roc:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_dt))

In [ ]:
# Save Decision Tree model
joblib.dump(best_dt, '../models/decision_tree_best.pkl')
print("Decision Tree model saved.")

---
# Part 4: Random Forest Modeling

### Investigation Prompts:
> - How does Random Forest differ from a single Decision Tree?
> - What is the effect of n_estimators on performance vs. computation?
> - Why might Random Forest be more robust than a single tree?

In [ ]:
# TODO: Define the hyperparameter grid for Random Forest
# Consider: n_estimators, max_depth, min_samples_split, max_features

rf_param_grid = {
    # YOUR CODE HERE
    # Hint: 'n_estimators': [50, 100, 200]
    # Hint: 'max_depth': [5, 10, 15, None]
    # Hint: 'max_features': ['sqrt', 'log2']
}

print(f"RF Parameter grid: {rf_param_grid}")

In [ ]:
# TODO: Perform GridSearchCV for Random Forest

rf = RandomForestClassifier(random_state=RANDOM_STATE)

rf_grid_search = GridSearchCV(
    # YOUR CODE HERE
)

# Fit
# YOUR CODE HERE

print(f"Best RF parameters: {rf_grid_search.best_params_}")
print(f"Best RF CV AUC-ROC: {rf_grid_search.best_score_:.4f}")

In [ ]:
# Evaluate Random Forest on test set
best_rf = rf_grid_search.best_estimator_

y_pred_rf = best_rf.predict(X_test)
y_proba_rf = best_rf.predict_proba(X_test)[:, 1]

rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_auc_roc = roc_auc_score(y_test, y_proba_rf)

print(f"\n=== Random Forest Results ===")
print(f"Test Accuracy: {rf_accuracy:.4f}")
print(f"Test AUC-ROC: {rf_auc_roc:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

In [ ]:
# Save Random Forest model
joblib.dump(best_rf, '../models/random_forest_best.pkl')
print("Random Forest model saved.")

---
# Part 5: eli5 for Decision Tree (Task 3_B)

## eli5 Investigation Prompts for Trees

> **Research these questions:**
> 
> 1. How does eli5 calculate feature importance for Decision Trees?
> 2. What does `eli5.show_prediction()` show for tree-based models vs. linear models?
> 3. How can you trace the decision path through a tree for a specific prediction?
> 4. Why might eli5 feature importance differ from sklearn's `feature_importances_`?

### 5.1 Feature Importance with eli5

In [ ]:
# TODO: Use eli5.show_weights() to display feature importance
# Hint: Pass feature_names for readability

# YOUR CODE HERE

In [ ]:
# TODO: Compare with sklearn's built-in feature_importances_
# Are they the same? Why or why not?

sklearn_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': best_dt.feature_importances_
}).sort_values('importance', ascending=False)

print("sklearn feature_importances_:")
print(sklearn_importance)

### 5.2 Explain Specific Predictions

In [ ]:
# Find indices for positive and negative examples
positive_idx = y_test[y_test == 1].index[0]
negative_idx = y_test[y_test == 0].index[0]

print(f"Positive example index: {positive_idx}")
print(f"Negative example index: {negative_idx}")

In [ ]:
# TODO: Explain prediction for POSITIVE example using eli5
# Hint: For trees, eli5.show_prediction() traces the decision path

print("=== Decision Tree Explanation for POSITIVE Example ===")
# YOUR CODE HERE

In [ ]:
# TODO: Explain prediction for NEGATIVE example

print("=== Decision Tree Explanation for NEGATIVE Example ===")
# YOUR CODE HERE

---
# Part 6: LIME for Random Forest (Task 3_C partial)

## LIME Investigation Prompts

> **Research these questions:**
> 
> 1. What does LIME stand for, and what is "local" interpretability?
> 2. What parameters does `LimeTabularExplainer` require and why?
> 3. What do the coefficients from `explain_instance()` represent?
> 4. What does a low R² value tell you about the LIME explanation?
> 5. How do you access `local_exp`, `intercept`, and `score` from an explanation?

### 6.1 Create LIME Explainer

In [ ]:
# TODO: Create a LimeTabularExplainer
# Hint: You need training_data, feature_names, class_names, mode

explainer = lime.lime_tabular.LimeTabularExplainer(
    # YOUR CODE HERE
    # training_data=X_train.values,
    # feature_names=feature_names,
    # class_names=['Survived', 'Death'],
    # mode='classification'
)

print("LIME explainer created.")

### 6.2 Explain Predictions with LIME

In [ ]:
# TODO: Explain a POSITIVE example prediction
# Use explain_instance() with the RF's predict_proba

pos_idx_loc = list(X_test.index).index(positive_idx)
pos_instance = X_test.iloc[pos_idx_loc].values

# YOUR CODE HERE
# exp_positive = explainer.explain_instance(...)

print("=== LIME Explanation for POSITIVE Example ===")
# exp_positive.show_in_notebook()  # or use as_list()

In [ ]:
# TODO: Access the LIME linear model details
# The project requires: coefficients, intercept, and R²

# YOUR CODE HERE
# Hint: exp_positive.local_exp gives feature contributions
# Hint: exp_positive.intercept gives the intercept
# Hint: exp_positive.score gives the R² of the local linear model

print("\nLIME Linear Model Details (Positive Example):")
# print(f"Intercept: {exp_positive.intercept}")
# print(f"R² Score: {exp_positive.score}")
# print(f"Coefficients: {exp_positive.local_exp}")

In [ ]:
# TODO: Explain a NEGATIVE example prediction

neg_idx_loc = list(X_test.index).index(negative_idx)
neg_instance = X_test.iloc[neg_idx_loc].values

print("=== LIME Explanation for NEGATIVE Example ===")
# YOUR CODE HERE

### 6.3 LIME Interpretation Summary

*Document your LIME findings:*

1. **R² Score Analysis:**
   - Positive example R²: ...
   - Negative example R²: ...
   - Is the local linear approximation trustworthy?

2. **Key Feature Contributions:**
   - Positive example top features: ...
   - Negative example top features: ...

3. **Comparison with eli5 (Decision Tree):**
   - How do LIME explanations for RF compare to eli5 for DT?
   - Any consistent features across both methods?

---
# Part 7: Individual Predictions for Task 4

In [ ]:
# TODO: Fill in the agreed indices from team discussion
POSITIVE_EXAMPLE_IDX = None  # Update with team-agreed index
NEGATIVE_EXAMPLE_IDX = None  # Update with team-agreed index

if POSITIVE_EXAMPLE_IDX is None:
    print("WARNING: Update example indices!")

In [ ]:
def predict_with_proba(model, X, idx, true_label, model_name):
    """Generate prediction output for Task 4 format."""
    pos = list(X.index).index(idx)
    sample = X.iloc[pos:pos+1]
    
    proba = model.predict_proba(sample)[0]
    pred = model.predict(sample)[0]
    
    print(f"{model_name} Prediction: {pred}")
    print(f"{model_name} Probabilities: [P(Survive)={proba[0]:.4f}, P(Death)={proba[1]:.4f}]")

if POSITIVE_EXAMPLE_IDX is not None:
    print("=== Positive Example (True Label: 1) ===")
    predict_with_proba(best_dt, X_test, POSITIVE_EXAMPLE_IDX, 1, "DT")
    predict_with_proba(best_rf, X_test, POSITIVE_EXAMPLE_IDX, 1, "RF")
    
    print("\n=== Negative Example (True Label: 0) ===")
    predict_with_proba(best_dt, X_test, NEGATIVE_EXAMPLE_IDX, 0, "DT")
    predict_with_proba(best_rf, X_test, NEGATIVE_EXAMPLE_IDX, 0, "RF")

---
# Part 8: Summary for Merge

### Decision Tree Results:

| Metric | Value |
|--------|-------|
| Best Hyperparameters | ... |
| Test Accuracy | ... |
| Test AUC-ROC | ... |

### Random Forest Results:

| Metric | Value |
|--------|-------|
| Best Hyperparameters | ... |
| Test Accuracy | ... |
| Test AUC-ROC | ... |

### Files Saved:
- `../models/decision_tree_best.pkl`
- `../models/random_forest_best.pkl`
- `../visualizations/continuous_distributions.png`
- `../visualizations/binary_distributions.png`

### Key Interpretability Findings:
1. eli5 (Decision Tree): ...
2. LIME (Random Forest): ...
3. R² assessment: ...